<a href="https://colab.research.google.com/github/tomonari-masada/course2026-nlp/blob/main/dimensionality_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dimensionality reduction

* GPUが使えるようにランタイムを設定しておく。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
#torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"DEVICE: {DEVICE}")

%config InlineBackend.figure_format = 'retina'

In [ ]:
from datasets import load_dataset

ds = load_dataset("ylecun/mnist")

In [ ]:
ds

In [ ]:
X_train = np.array(ds["train"]["image"])
X_train = X_train.reshape(X_train.shape[0], -1)
X_train = X_train / X_train.max()
X_test = np.array(ds["test"]["image"])
X_test = X_test.reshape(X_test.shape[0], -1)
X_test = X_test / X_train.max()

y_train = ds["train"]["label"]
y_test = ds["test"]["label"]

In [ ]:
# global centering
train_mean = X_train.mean(axis=0)
X_train_centered = X_train - train_mean
X_test_centered = X_test - train_mean

# local centering
X_train = X_train_centered - X_train_centered.mean(axis=1).reshape(-1, 1)
X_test = X_test_centered - X_test_centered.mean(axis=1).reshape(-1, 1)

In [ ]:
X_train.shape, X_test.shape

## 次元圧縮手法によるデータの再構成(reconstruction)

* 画像を複数描画する関数を定義しておく。

In [ ]:
def plot_gallery(title, images, n_col=3, n_row=3, cmap=plt.cm.gray):
  plt.figure(figsize=(2. * n_col, 2.26 * n_row))
  plt.suptitle(title, size=16)
  for i, comp in enumerate(images):
    plt.subplot(n_row, n_col, i + 1)
    vmax = max(comp.max(), -comp.min())
    plt.imshow(
        comp.reshape(28, 28),
        cmap=cmap,
        interpolation="nearest",
        vmin=-vmax,
        vmax=vmax,
    )
    plt.xticks([])
    plt.yticks([])
  plt.subplots_adjust(0.01, 0.05, 0.99, 0.93, 0.04, 0.);

In [ ]:
plot_gallery("sign mnist", X_train[:9], n_row=3, n_col=3)

In [ ]:
n_samples, sample_dim = X_train.shape
n_test_samples = X_test.shape[0]

print(f"n_samples: {n_samples}, sample_dim: {sample_dim}, n_test_samples: {n_test_samples}")

In [ ]:
n_components = 100

X_train_torch = torch.tensor(X_train, dtype=torch.float, device=DEVICE)

autoencoder = torch.nn.Sequential(
    torch.nn.Linear(sample_dim, n_components, bias=False, device=DEVICE),
    torch.nn.ReLU(),
    torch.nn.Linear(n_components, sample_dim, bias=False, device=DEVICE),
)
autoencoder = autoencoder.to(DEVICE)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

batch_size = 1000
epochs = 500

n_steps = 0
for epoch in range(1, epochs + 1):
  random_indices = torch.randperm(n_samples)
  for i in range(0, n_samples, batch_size):
    indices = random_indices[i:i+batch_size]
    optimizer.zero_grad(set_to_none=True)
    reconstruction = autoencoder(X_train_torch[indices])
    loss = criterion(reconstruction, X_train_torch[indices])
    loss.backward()
    optimizer.step()

    n_steps += 1
    if n_steps % 100 == 0:
      print(f"epoch {epoch} | step {n_steps} | loss {loss.item():.4e}")


In [ ]:
for name, param in autoencoder.named_parameters():
  print(f"{name}: {param.shape}")

In [ ]:
autoencoder[2].weight

In [ ]:
components = autoencoder[2].weight.detach().cpu().numpy().transpose()

In [ ]:
n_col = 6
n_row = n_components // n_col + (n_components % n_col != 0)
plot_gallery(
    f"{n_components} components",
    components[:n_components],
    n_row=n_row,
    n_col=n_col,
)

### テストセットに含まれる画像の再構成

In [ ]:
X_test_recon = autoencoder(torch.tensor(X_test, dtype=torch.float, device=DEVICE)).detach().cpu().numpy()

In [ ]:
n_recon_images = 8
indices = np.random.randint(X_test.shape[0], size=n_recon_images)
plot_gallery("original test data", X_test[indices], n_row=1, n_col=n_recon_images)
plot_gallery("reconstructed test data", X_test_recon[indices], n_row=1, n_col=n_recon_images)